# [Traces] T4 + Qwen3-4B + AIME: Bulk Strategy Tuning

## Purpose
Generate **11,000+ traces** on AIME problems using a weak but fast model. These traces are used to **tune selection strategies** (entropy weighting, vote bonuses, thresholds) that will later transfer to stronger models.

## Why This Configuration?
- **T4 GPU**: Free 30h/week quota, sufficient for small models
- **Qwen3-4B**: Fast (~15s/sample), decent math (~40% AIME), generates diverse answers
- **AIME dataset**: 2,250 problems with known answers - perfect for strategy tuning

## Why Weak Model + Many Problems?
We're training a **selection strategy**, not the LLM. The strategy learns:
- "Low entropy → more likely correct"
- "More votes → more likely correct"
- Optimal weighting between signals

These patterns **transfer across models**. A strategy tuned on Qwen3-4B works on gpt-oss-120b.

## Output
```
/kaggle/working/traces/
    problem_{id}.json  -- Per-problem traces with entropy, answers, code executions
    summary.json       -- Overall accuracy and timing
    config.json        -- Generation parameters
```

## Expected Results
- ~2,000 problems × 5 samples = **10,000+ traces**
- Runtime: ~8-9 hours on T4
- Accuracy: ~40% (expected for weak model on olympiad math)

In [ ]:
# Cell 1: Install vLLM
%pip install -q vllm transformers accelerate

In [ ]:
# Cell 2: Imports
import warnings; warnings.filterwarnings('ignore')
import os, sys, json, re, math, time, threading, queue, subprocess
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, Dict, List, Any
import pandas as pd

# Set environment
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
# Cell 3: Configuration
class CFG:
    # Model
    model_name = 'Qwen/Qwen3-4B'  # Will download from HF
    model_path = '/kaggle/input/qwen-3/transformers/qwen3-4b/1'  # Kaggle model path
    
    # Prompts
    system_prompt = (
        'You are a math competition solver. Solve the problem step by step. '
        'You can use Python code in ```python``` blocks which will be executed. '
        'Put your final integer answer in \\boxed{}.'
    )
    
    # Generation
    n_samples = 5
    max_turns = 8  # Fewer turns for speed
    max_tokens = 4096
    temperature = 0.8
    top_p = 0.95
    
    # vLLM
    gpu_memory_utilization = 0.90
    max_model_len = 8192
    
    # Timing
    problem_timeout = 180  # 3 min per problem (fast model)
    code_timeout = 10
    
    # Output
    output_dir = '/kaggle/working/traces'
    
    seed = 42

print(f"Config: {CFG.model_name}, {CFG.n_samples} samples, {CFG.max_turns} turns")

In [ ]:
# Cell 4: Load AIME Dataset
def load_aime_problems():
    """Load AIME problems from Kaggle dataset."""
    path = '/kaggle/input/aime-problem-set-1983-2024/AIME_Dataset_1983_2024.csv'
    df = pd.read_csv(path)
    
    # Normalize columns
    df.columns = df.columns.str.lower().str.strip()
    
    # Create unique ID
    if 'id' not in df.columns:
        df['id'] = df.apply(lambda r: f"aime_{r.get('year', 0)}_{r.get('problem number', r.name)}", axis=1)
    
    # Find problem column
    prob_col = next((c for c in df.columns if 'problem' in c.lower() and 'number' not in c.lower()), None)
    if prob_col and prob_col != 'problem':
        df['problem'] = df[prob_col]
    
    # Find answer column
    ans_col = next((c for c in df.columns if 'answer' in c.lower()), None)
    if ans_col and ans_col != 'answer':
        df['answer'] = df[ans_col]
    
    print(f"Loaded {len(df)} AIME problems")
    print(f"Columns: {list(df.columns)}")
    print(f"Years: {df['year'].min() if 'year' in df.columns else 'N/A'} - {df['year'].max() if 'year' in df.columns else 'N/A'}")
    
    return df

df = load_aime_problems()
print(f"\nFirst problem: {df.iloc[0]['problem'][:100]}...")

In [ ]:
# Cell 5: Simple Code Sandbox
import subprocess
import tempfile

def execute_code(code: str, timeout: int = 10) -> str:
    """Execute Python code and return output."""
    # Add common imports
    full_code = "import math, itertools, functools\nfrom fractions import Fraction\n" + code
    
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(full_code)
            f.flush()
            result = subprocess.run(
                ['python', f.name],
                capture_output=True,
                text=True,
                timeout=timeout
            )
            os.unlink(f.name)
            
            if result.returncode != 0:
                return f"[ERROR] {result.stderr[:500]}"
            return result.stdout[:2000] or "[No output]"
    except subprocess.TimeoutExpired:
        return "[ERROR] Timeout"
    except Exception as e:
        return f"[ERROR] {str(e)[:200]}"

# Test
print(execute_code("print(2**10)"))

In [ ]:
# Cell 6: Initialize vLLM
from vllm import LLM, SamplingParams

# Check if Kaggle model path exists, otherwise use HF
if os.path.exists(CFG.model_path):
    model_path = CFG.model_path
    print(f"Using Kaggle model: {model_path}")
else:
    model_path = CFG.model_name
    print(f"Using HuggingFace model: {model_path}")

llm = LLM(
    model=model_path,
    gpu_memory_utilization=CFG.gpu_memory_utilization,
    max_model_len=CFG.max_model_len,
    trust_remote_code=True,
    seed=CFG.seed,
)

print(f"Model loaded: {model_path}")

In [ ]:
# Cell 7: Helper Functions
def extract_answer(text: str) -> Optional[int]:
    """Extract integer answer from \\boxed{}."""
    patterns = [
        r'\\boxed\s*\{\s*([0-9]+)\s*\}',
        r'answer\s*(?:is|=)\s*([0-9]+)',
        r'final\s+answer\s*:?\s*([0-9]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            try:
                val = int(matches[-1])
                if 0 <= val <= 999:  # AIME answers are 0-999
                    return val
            except ValueError:
                pass
    return None

def extract_code_blocks(text: str) -> List[str]:
    """Extract Python code blocks."""
    pattern = r'```(?:python)?\s*\n(.*?)```'
    return re.findall(pattern, text, re.DOTALL | re.IGNORECASE)

def compute_entropy(logprobs: List[Dict]) -> float:
    """Compute mean entropy from logprobs."""
    if not logprobs:
        return float('inf')
    total, count = 0.0, 0
    for lp_dict in logprobs:
        if isinstance(lp_dict, dict) and lp_dict:
            ent = sum(-math.exp(lp) * math.log2(max(math.exp(lp), 1e-10))
                     for lp in lp_dict.values() if lp is not None)
            total += ent
            count += 1
    return total / count if count else float('inf')

print("Helpers loaded")

In [ ]:
# Cell 8: Single Attempt Function
def solve_once(problem_text: str, seed: int) -> Dict[str, Any]:
    """Run one TIR attempt on a problem."""
    
    messages = [
        {"role": "system", "content": CFG.system_prompt},
        {"role": "user", "content": problem_text}
    ]
    
    # Build prompt using chat template
    tokenizer = llm.get_tokenizer()
    
    full_text = ""
    all_logprobs = []
    code_executions = []
    answer = None
    answer_source = None
    turns_used = 0
    total_tokens = 0
    last_code_output = None
    
    sampling_params = SamplingParams(
        temperature=CFG.temperature,
        top_p=CFG.top_p,
        max_tokens=CFG.max_tokens,
        seed=seed,
        logprobs=5,
    )
    
    for turn in range(CFG.max_turns):
        turns_used = turn + 1
        
        # Format prompt
        prompt = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # Generate
        outputs = llm.generate([prompt], sampling_params)
        response = outputs[0].outputs[0]
        text = response.text
        total_tokens += len(response.token_ids)
        
        # Collect logprobs
        if response.logprobs:
            for lp in response.logprobs:
                if lp:
                    all_logprobs.append({k: v.logprob for k, v in lp.items()})
        
        full_text += text + "\n"
        messages.append({"role": "assistant", "content": text})
        
        # Check for answer
        ans = extract_answer(text)
        if ans is not None:
            answer = ans
            answer_source = "boxed"
            break
        
        # Execute code if present
        code_blocks = extract_code_blocks(text)
        if code_blocks:
            outputs_list = []
            for code in code_blocks:
                output = execute_code(code, CFG.code_timeout)
                is_error = '[ERROR]' in output
                if not is_error:
                    last_code_output = output
                outputs_list.append(output)
                code_executions.append({
                    'turn': turn,
                    'code': code[:500],
                    'output': output[:500],
                    'is_error': is_error
                })
            
            # Add code output to conversation
            exec_text = "\n".join(outputs_list)
            messages.append({
                "role": "user", 
                "content": f"Code output:\n```\n{exec_text}\n```\nContinue solving. Put your final answer in \\boxed{{}}."
            })
        else:
            # No code, prompt to continue
            messages.append({
                "role": "user",
                "content": "Continue solving. Put your final answer in \\boxed{}."
            })
    
    # Fallback: extract from code output
    if answer is None and last_code_output:
        for num in re.findall(r'\b(\d{1,3})\b', last_code_output):
            val = int(num)
            if 0 <= val <= 999:
                answer = val
                answer_source = "code_fallback"
                break
    
    entropy = compute_entropy(all_logprobs)
    if answer_source == "code_fallback":
        entropy = max(entropy, 8.0)  # Penalize fallback
    
    return {
        'answer': answer,
        'answer_source': answer_source,
        'entropy': entropy,
        'turns_used': turns_used,
        'total_tokens': total_tokens,
        'code_executions': code_executions,
        'n_python_calls': len(code_executions),
        'n_python_errors': sum(1 for c in code_executions if c['is_error']),
    }

print("Solver function defined")

In [ ]:
# Cell 9: Main Trace Generation Loop
def generate_all_traces(df: pd.DataFrame) -> Dict:
    """Generate traces for all problems."""
    
    os.makedirs(CFG.output_dir, exist_ok=True)
    
    # Save config
    config = {
        'model': CFG.model_name,
        'n_samples': CFG.n_samples,
        'max_turns': CFG.max_turns,
        'temperature': CFG.temperature,
        'n_problems': len(df),
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(f"{CFG.output_dir}/config.json", 'w') as f:
        json.dump(config, f, indent=2)
    
    all_results = []
    total_correct = 0
    start_time = time.time()
    
    for idx, row in df.iterrows():
        prob_id = str(row.get('id', idx))
        problem_text = row['problem']
        ground_truth = int(row['answer']) if pd.notna(row.get('answer')) else None
        
        print(f"\n{'='*50}")
        print(f"Problem {idx+1}/{len(df)} [{prob_id}]")
        if ground_truth is not None:
            print(f"Ground truth: {ground_truth}")
        
        prob_start = time.time()
        attempts = []
        
        for i in range(CFG.n_samples):
            seed = CFG.seed + idx * 100 + i * 7
            t0 = time.time()
            
            try:
                result = solve_once(problem_text, seed)
                result['attempt_idx'] = i
                result['seed'] = seed
                result['wall_time_s'] = round(time.time() - t0, 2)
                result['prompt_type'] = 'reasoning'
            except Exception as e:
                result = {
                    'attempt_idx': i,
                    'seed': seed,
                    'answer': None,
                    'answer_source': None,
                    'entropy': float('inf'),
                    'error': str(e)[:200],
                    'wall_time_s': round(time.time() - t0, 2),
                    'prompt_type': 'reasoning',
                }
            
            attempts.append(result)
            ans = result.get('answer')
            ent = result.get('entropy', float('inf'))
            print(f"  Sample {i+1}/{CFG.n_samples}: answer={ans}, entropy={ent:.3f}")
        
        # Majority vote
        valid = [a['answer'] for a in attempts if a['answer'] is not None]
        if valid:
            default_answer = Counter(valid).most_common(1)[0][0]
        else:
            default_answer = 0
        
        is_correct = default_answer == ground_truth if ground_truth is not None else None
        if is_correct:
            total_correct += 1
        
        tag = '✓' if is_correct else ('✗' if is_correct is False else '?')
        print(f"  >> [{tag}] Default={default_answer}, Expected={ground_truth}")
        
        # Save trace
        trace = {
            'problem_id': prob_id,
            'problem_text': problem_text,
            'ground_truth': ground_truth,
            'wall_time_s': round(time.time() - prob_start, 2),
            'attempts': attempts,
            'default_answer': default_answer,
            'default_method': 'majority_vote',
            'default_votes': dict(Counter(valid)),
        }
        
        with open(f"{CFG.output_dir}/problem_{prob_id}.json", 'w') as f:
            json.dump(trace, f, indent=2, default=lambda x: str(x) if isinstance(x, float) and (math.isinf(x) or math.isnan(x)) else x)
        
        all_results.append({
            'problem_id': prob_id,
            'ground_truth': ground_truth,
            'default_answer': default_answer,
            'correct': is_correct,
        })
        
        # Progress
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed * 3600
        print(f"  Progress: {idx+1}/{len(df)} | {total_correct} correct | {rate:.0f} prob/hr")
    
    # Summary
    total_with_answer = sum(1 for r in all_results if r['ground_truth'] is not None)
    accuracy = total_correct / total_with_answer if total_with_answer else 0
    
    summary = {
        'model': CFG.model_name,
        'n_samples': CFG.n_samples,
        'correct': total_correct,
        'total': total_with_answer,
        'accuracy': round(accuracy, 4),
        'total_time_s': round(time.time() - start_time, 1),
        'per_problem': all_results,
    }
    
    with open(f"{CFG.output_dir}/summary.json", 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n{'#'*50}")
    print(f"COMPLETE: {total_correct}/{total_with_answer} ({accuracy*100:.1f}%)")
    print(f"Time: {(time.time()-start_time)/3600:.1f} hours")
    print(f"Traces: {CFG.output_dir}")
    
    return summary

print("Generator function defined")

In [ ]:
# Cell 10: Run Generation
summary = generate_all_traces(df)

In [ ]:
# Cell 11: Show Results
print(f"\nFinal Results:")
print(f"  Problems: {summary['total']}")
print(f"  Correct: {summary['correct']}")
print(f"  Accuracy: {summary['accuracy']*100:.1f}%")
print(f"  Time: {summary['total_time_s']/3600:.1f} hours")
print(f"\nTraces saved to: {CFG.output_dir}")
print(f"Files: {len(os.listdir(CFG.output_dir))}")